# Lab 1 — Vector Spaces in Python
**Linear Algebra · UATX**

Goal: turn the abstract axioms into Python checks, probe subspaces, visualise span, and explore the isomorphism $P_2 \cong \mathbb{R}^3$.

**Tasks**
1. Write a stochastic axiom-checker and test it on $\mathbb{R}^3$ and $P_2$.
2. Test the subspace criterion on concrete examples in $\mathbb{R}^3$.
3. Visualise span and linear combinations in $\mathbb{R}^2$ and $\mathbb{R}^3$.
4. Verify the isomorphism $P_2 \cong \mathbb{R}^3$ computationally.
5. Test whether two subspaces form a direct sum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import null_space

np.random.seed(42)

## §1  Stochastic Axiom Checker

A vector space $(V, +, \cdot)$ over $\mathbb{R}$ satisfies eight axioms. We can *test* them on randomly sampled vectors: a single counterexample disproves an axiom; many passing samples give strong evidence of validity.

In [ ]:
def check_vector_space_axioms(sample, add, scale, zero, neg, n=300, tol=1e-9):
    """
    Stochastically test 8 vector space axioms.

    Parameters
    ----------
    sample : () -> v        random element of V
    add    : (u, v) -> v    addition in V
    scale  : (a, v) -> v    scalar multiplication
    zero   :                zero element of V
    neg    : v -> v         additive inverse
    """
    close = lambda x, y: np.allclose(x, y, atol=tol)
    results = {}

    for _ in range(n):
        u, v, w = sample(), sample(), sample()
        a, b = np.random.randn(), np.random.randn()

        axioms = {
            'A2 commutativity      u+v = v+u'       : close(add(u,v), add(v,u)),
            'A3 associativity (u+v)+w = u+(v+w)'    : close(add(add(u,v),w), add(u,add(v,w))),
            'A4 zero element       u+0 = u'          : close(add(u, zero), u),
            'A5 additive inverse   u+(-u) = 0'       : close(add(u, neg(u)), zero),
            'A6 unit scalar        1*u = u'           : close(scale(1.0, u), u),
            'A7 dist. over vec+    a*(u+v)=a*u+a*v'  : close(scale(a, add(u,v)),
                                                              add(scale(a,u), scale(a,v))),
            'A8 dist. over scal+   (a+b)*u=a*u+b*u'  : close(scale(a+b, u),
                                                              add(scale(a,u), scale(b,u))),
        }
        for name, ok in axioms.items():
            results[name] = results.get(name, True) and ok

    for name, ok in results.items():
        print(f'  {"PASS" if ok else "FAIL"}  {name}')
    return all(results.values())

In [ ]:
# Test on R^3
print('R^3:')
check_vector_space_axioms(
    sample=lambda: np.random.randn(3),
    add=lambda u,v: u+v,
    scale=lambda a,v: a*v,
    zero=np.zeros(3),
    neg=lambda v: -v
)

In [ ]:
# Test on P_2: polynomials a0 + a1*x + a2*x^2 represented as coefficient vectors [a0, a1, a2]
print('P_2 (represented as coefficient vectors in R^3):')
check_vector_space_axioms(
    sample=lambda: np.random.randn(3),
    add=lambda u,v: u+v,
    scale=lambda a,v: a*v,
    zero=np.zeros(3),
    neg=lambda v: -v
)

In [ ]:
# Non-example: test an affine subspace {x in R^3 : x[0] + x[1] = 1} (not through origin)
print('Affine plane {x : x[0]+x[1]=1}  (should FAIL A4/A5):')
check_vector_space_axioms(
    sample=lambda: np.array([t:=np.random.randn(), 1-t, np.random.randn()]),
    add=lambda u,v: u+v,
    scale=lambda a,v: a*v,
    zero=np.array([0.5, 0.5, 0.0]),   # 'zero' of this set? there is none that works
    neg=lambda v: -v
)

## §2  The Subspace Test

A subset $W \subseteq V$ is a **subspace** iff for all $u, v \in W$ and $\alpha, \beta \in \mathbb{R}$: $\alpha u + \beta v \in W$.

We encode $W$ by a sampler `sample_from()` and a membership oracle `in_set()`.

In [ ]:
def is_subspace(sample_fn, in_set_fn, n=500):
    """Stochastic subspace test."""
    for _ in range(n):
        u, v = sample_fn(), sample_fn()
        a, b = np.random.randn(), np.random.randn()
        if not in_set_fn(a*u + b*v):
            return False
    return True


# (a) {x in R^3 : x[0] + x[1] = 0}  — a plane through the origin
sample_a = lambda: np.array([t:=np.random.randn(), -t, np.random.randn()])
in_a = lambda v: abs(v[0] + v[1]) < 1e-9
print('(a) {x : x0+x1=0}:', 'SUBSPACE' if is_subspace(sample_a, in_a) else 'NOT subspace')

# (b) {x : x[0]*x[1] = 0}  — union of two planes; closed under scalar mult but NOT addition
u_b = np.array([1., 0., 0.])
v_b = np.array([0., 1., 0.])
print('(b) {x : x0*x1=0}: counterexample u+v =', u_b+v_b, '-> x0*x1 =', (u_b+v_b)[0]*(u_b+v_b)[1])

# (c) Unit sphere {x : ||x||=1}  — zero is not in it
print('(c) Unit sphere: zero not in it, so NOT a subspace')

# (d) ker(A): always a subspace
A = np.array([[1., 2., -1.], [2., 4., -2.]])
K = null_space(A)
def sample_kernel():
    coeffs = np.random.randn(K.shape[1])
    return K @ coeffs
in_kernel = lambda v: np.linalg.norm(A @ v) < 1e-8
print('(d) ker(A):', 'SUBSPACE' if is_subspace(sample_kernel, in_kernel) else 'NOT subspace')

## §3  Span and Linear Combinations

The **span** of $\{v_1, \ldots, v_k\}$ is the set of all linear combinations. We visualise it by sampling many random combinations $\alpha_1 v_1 + \cdots + \alpha_k v_k$ with $\alpha_i \sim N(0,1)$.

In [ ]:
def sample_span(vectors, n=1000):
    """Sample n random linear combinations of the given vectors."""
    V = np.column_stack(vectors)
    coeffs = np.random.randn(V.shape[1], n)
    return (V @ coeffs).T


fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# One vector -> line
pts = sample_span([np.array([1., 2.])])
axes[0].scatter(pts[:,0], pts[:,1], s=2, alpha=0.3)
axes[0].set_title('span{(1,2)} — a line in $\\mathbb{R}^2$'); axes[0].set_aspect('equal'); axes[0].grid(True)

# Two independent vectors -> all of R^2
pts = sample_span([np.array([1.,0.]), np.array([0.,1.])])
axes[1].scatter(pts[:,0], pts[:,1], s=2, alpha=0.1)
axes[1].set_title('span{$e_1$,$e_2$} = $\\mathbb{R}^2$'); axes[1].set_aspect('equal'); axes[1].grid(True)

# Two collinear vectors -> still a line
pts = sample_span([np.array([1.,2.]), np.array([2.,4.])])
axes[2].scatter(pts[:,0], pts[:,1], s=2, alpha=0.3)
axes[2].set_title('span{(1,2),(2,4)} — line (dependent!)'); axes[2].set_aspect('equal'); axes[2].grid(True)

plt.tight_layout(); plt.show()

In [ ]:
# 3D: span of two independent vectors = a plane
w1 = np.array([1., 0., 0.])
w2 = np.array([0., 1., 1.])
pts3 = sample_span([w1, w2], n=1500)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts3[:,0], pts3[:,1], pts3[:,2], s=2, alpha=0.15)
# draw the two basis vectors
for v, lbl in [(w1,'$w_1$'),(w2,'$w_2$')]:
    ax.quiver(0,0,0,*v, color='red', linewidth=2)
    ax.text(*(v*1.1), lbl, fontsize=12, color='red')
ax.set_title('span{$w_1$, $w_2$} — a plane in $\\mathbb{R}^3$')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.tight_layout(); plt.show()

print('dim(span) = rank of [w1|w2] =', np.linalg.matrix_rank(np.column_stack([w1, w2])))

## §4  The Isomorphism $P_2 \cong \mathbb{R}^3$

The map $\phi: P_2 \to \mathbb{R}^3$, $\phi(a_0 + a_1 x + a_2 x^2) = (a_0, a_1, a_2)^\top$ is a **linear isomorphism**: it preserves addition and scalar multiplication. Two isomorphic spaces are "the same" up to relabelling.

In [ ]:
def poly_eval(coeffs, x):
    """Evaluate polynomial with given coefficients at x."""
    return sum(c * x**i for i, c in enumerate(coeffs))

# Two random polynomials
p = np.random.randn(3)
q = np.random.randn(3)
a = np.random.randn()

# Verify phi preserves addition: phi(p+q) = phi(p) + phi(q)
# Since phi IS the identity on coefficients, this is just numpy addition:
print('phi preserves addition:        ', np.allclose(p + q, p + q))    # trivially true

# The real check: evaluate at the same point using two methods
x0 = 2.5
val_direct = poly_eval(p + q, x0)
val_via_phi = poly_eval(p, x0) + poly_eval(q, x0)
print(f'(p+q)(2.5) via direct eval:    {val_direct:.8f}')
print(f'p(2.5) + q(2.5) via phi:       {val_via_phi:.8f}')
print(f'Match: {np.isclose(val_direct, val_via_phi)}')

# Verify phi preserves scalar mult: phi(a*p)(x) = a * phi(p)(x)
val1 = poly_eval(a * p, x0)
val2 = a * poly_eval(p, x0)
print(f'\n(a*p)(2.5) = {val1:.8f},   a*p(2.5) = {val2:.8f},   match: {np.isclose(val1, val2)}')

In [ ]:
# Visualise: plot p, q, and p+q to confirm linearity
xs = np.linspace(-2, 2, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, poly_eval(p[0:1+1+1], xs) if False else np.polyval(p[::-1], xs), label='$p(x)$')
ax.plot(xs, np.polyval(q[::-1], xs), label='$q(x)$')
ax.plot(xs, np.polyval((p+q)[::-1], xs), '--', lw=2, label='$(p+q)(x)$')
ax.plot(xs, np.polyval(p[::-1], xs) + np.polyval(q[::-1], xs), ':', lw=2, label='$p(x)+q(x)$')
ax.legend(); ax.set_title('Linearity of $\\phi$: $(p+q)(x) = p(x)+q(x)$'); ax.grid(True)
plt.tight_layout(); plt.show()

## Direct Sum Test

Subspaces $U, W \subseteq \mathbb{R}^n$ form a **direct sum** $\mathbb{R}^n = U \oplus W$ iff:
- $U \cap W = \{0\}$ (intersection trivial), i.e. $\mathrm{rank}([B_U | B_W]) = \mathrm{rank}(B_U) + \mathrm{rank}(B_W)$
- $U + W = \mathbb{R}^n$ (span full), i.e. $\mathrm{rank}([B_U | B_W]) = n$

In [ ]:
def direct_sum_test(B_U, B_W):
    n = B_U.shape[0]
    combined = np.hstack([B_U, B_W])
    r_comb = np.linalg.matrix_rank(combined)
    r_U    = np.linalg.matrix_rank(B_U)
    r_W    = np.linalg.matrix_rank(B_W)
    trivial_intersection = (r_comb == r_U + r_W)
    full_span            = (r_comb == n)
    print(f'  dim(U)={r_U}, dim(W)={r_W}, dim(U+W)={r_comb}, n={n}')
    print(f'  U ∩ W = {{0}}: {trivial_intersection}')
    print(f'  U + W = R^n:  {full_span}')
    print(f'  Direct sum:   {trivial_intersection and full_span}')

# R^3 = span{e1,e2} ⊕ span{e3}
print('Example 1: span{e1,e2} and span{e3} in R^3')
direct_sum_test(np.eye(3)[:, :2], np.eye(3)[:, 2:])

# R^3 != span{e1,e2} ⊕ span{e2,e3}  (they share e2)
print('\nExample 2: span{e1,e2} and span{e2,e3} in R^3')
direct_sum_test(np.eye(3)[:, :2], np.eye(3)[:, 1:])